In [ ]:

import copy
import math
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, Dataset
import matplotlib.pyplot as plt
# import flwr_datasets
from datasets import concatenate_datasets
from collections import OrderedDict


In [ ]:
import random, os
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(45)  # call before creating model, dataset, etc.
g = torch.Generator().manual_seed(42)


In [ ]:
from collections import defaultdict

In [ ]:
def generate_failure_matrix(ne,nh,s):
    failure_matrix = [[0]*nh for _ in range(ne)]
    failure_dict = defaultdict(list)
    for i in range(ne):
        failed_connections = tuple(random.sample(range(0,nh),s))
        failure_dict[failed_connections].append(i)
        for j in failed_connections:
            failure_matrix[i][j]=1
    return failure_matrix,dict(failure_dict)

In [ ]:
def generate_full_rank_matrix_with_identity(n, k, max_int=100):
    identity = [[int(i == j) for j in range(n)] for i in range(n)]
    
    while True:
        extra_rows = [[int(np.random.randint(0, max_int + 1)) for _ in range(n)] for _ in range(k)]
        candidate_matrix = identity + extra_rows
        if np.linalg.matrix_rank(np.array(candidate_matrix)) == n:
            return candidate_matrix

In [ ]:
num_helpers = 5
num_clients = 20
num_erasures = 2
NUM_ROUNDS = 5
GLOBAL_FAILURE_MATRIX,GLOBAL_FAILURE_DICT = generate_failure_matrix(num_clients,num_helpers,num_erasures)
MDS_CODE = [[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,2,3]]
MDS_CODE = generate_full_rank_matrix_with_identity(num_helpers-num_erasures,num_erasures)
MDS_CODE = torch.tensor(MDS_CODE, dtype=torch.float32)
MDS_CODE_NP = np.array(MDS_CODE)

In [ ]:
print(MDS_CODE)

In [ ]:
GLOBAL_FAILURE_MATRIX

In [ ]:
GLOBAL_FAILURE_MATRIX

In [ ]:
GLOBAL_FAILURE_DICT

In [ ]:
from collections import defaultdict
payload_dict = defaultdict(list)

In [ ]:
# class FEMNISTNet(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv1 = nn.Conv2d(1, 32, 7, padding=3)
#         self.act = nn.ReLU()
#         self.pool = nn.MaxPool2d(2, 2)
#         self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
#         self.fc = nn.Linear(64 * 7 * 7, 62)   # 62 classes in FEMNIST

#     def forward(self, x):
#         x = x.reshape(-1, 1, 28, 28)
#         x = self.pool(self.act(self.conv1(x)))
#         x = self.pool(self.act(self.conv2(x)))
#         x = x.flatten(1)
#         return self.fc(x)

In [ ]:

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_size=28*28, num_classes=62):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        x = x.view(-1, 28*28)  # Flatten the input
        out = self.linear(x)
        return out


In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim=784, num_classes=62):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten 28x28
        return self.fc(x)

In [ ]:
metadata_of_logistic_regression = {
    ('fc.weight', (62, 784), 62*784),
    ('fc.bias', (62,), 62)
}

In [ ]:
# metadata_of_femnistnet = [
#     ('conv1.weight', (32, 1, 7, 7), 1568),
#     ('conv1.bias', (32,), 32),
#     ('conv2.weight', (64, 32, 3, 3), 18432),
#     ('conv2.bias', (64,), 64),
#     ('fc.weight', (62, 3136), 194432),
#     ('fc.bias', (62,), 62),
# ]

In [ ]:
def flatten_state_dict(state_dict,metadata=metadata_of_logistic_regression):
    flat_list = []
    for key, shape, numel in metadata:
        flat_list.append(state_dict[key].view(-1))
    return torch.cat(flat_list)

def split_tensor(tensor, num_parts):
    n = tensor.size(0)
    tensor = torch.cat([tensor, torch.zeros(num_parts - n % num_parts, dtype=tensor.dtype)])
    n = tensor.size(0)
    split_tensors = torch.split(tensor, n // num_parts)
    
    return torch.stack(split_tensors)

def rebuild_state_dict_from_flat(flat_list,metadata=metadata_of_logistic_regression):
    new_state_dict = OrderedDict()
    count = 0
    for key, shape, numel in metadata:
        new_state_dict[key] = flat_list[count:count+numel].view(shape)
        count += numel
    return new_state_dict

def rebuild_flat_tensor_from_pieces(tensor_pieces,len_tensor=48670):
    final_piece = torch.cat(tensor_pieces)
    final_piece = final_piece[:len_tensor]
    return final_piece

def payload(tensor):
    if tensor is None:
        return 0
    return tensor.nbytes

In [ ]:
# newnet = FEMNISTNet()
# for name, param in newnet.named_parameters():
#     print(name, param.shape)
# check = flatten_state_dict(newnet.state_dict())
# print(split_tensor(check,3))
# print(MDS_CODE.shape , split_tensor(check,3).shape)
# res = MDS_CODE @ split_tensor(check,3)
# print(res)
# print(res.shape)
# recovered = rebuild_state_dict_from_flat(check)
# print()
# for name in recovered:
#     print(name,recovered[name].shape)

In [ ]:
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner, GroupedNaturalIdPartitioner
from datasets import ClassLabel
from torch.utils.data import DataLoader

class HFWrapper(Dataset):
    def __init__(
        self, hf_dataset, image_key="image", label_key="character",
    ):
        self.hf = hf_dataset
        self.image_key = image_key
        self.label_key = label_key
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, idx):
        item = self.hf[idx]
        img = item[self.image_key]
        label = item[self.label_key]
        img = self.transform(img)
        return img, label

def make_loader(hf_dataset, batch_size=32, shuffle=False):
    wrapped = HFWrapper(hf_dataset)
    return DataLoader(
        wrapped,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=g)

partitioner = GroupedNaturalIdPartitioner(partition_by='writer_id',group_size=10)
dataset = FederatedDataset(
    dataset="flwrlabs/femnist", partitioners={"train": partitioner}
)
partition  = dataset.load_partition(0,'train')
partition.set_format(type='torch',columns=['character'])

client_subsets = []
test_datasubsets = []
for i in range(num_clients):
    curr_part = dataset.load_partition(partition_id=i, split="train")
    split_data = curr_part.train_test_split(test_size=0.2, seed=42)    
    assert len(split_data['test'])>0 and len(split_data['train'])>0
    client_subsets.append(split_data["train"])
    test_datasubsets.append(split_data["test"])


train_data = concatenate_datasets(client_subsets)
train_loader = make_loader(train_data, batch_size=256, shuffle=True)
test_data = concatenate_datasets(test_datasubsets)
test_loader = make_loader(test_data,batch_size=256,shuffle=False)


In [ ]:
for i, train_subset in enumerate(client_subsets):
    print(f"Client {i} train subset size: {len(train_subset)}")

In [ ]:
tot_size = 0
for i, train_subset in enumerate(client_subsets):
    tot_size += len(train_subset)


In [ ]:
class Helper(OrderedDict):
    def __init__(self,hid):
        super().__init__()
        self.hid = hid
    def send_model_to_master(self):
        pass

In [ ]:
helpers = [Helper(hid=i) for i in range(num_helpers)]

In [ ]:
class Client:
    def __init__(self,cid, dataset, device="cpu"):
        self.cid = cid
        self.dataset = dataset
        self.device = device
        self.model_state = None
        self.len_dataset = len(self.dataset)
        self.split_weights = None

    def train_local(self, global_model, epochs=1, batch_size=32, lr=0.01):
        model = copy.deepcopy(global_model).to(self.device)
        loader = make_loader(self.dataset, batch_size=batch_size, shuffle=True)
        optimizer = optim.SGD(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        model.train()
        for _ in range(epochs):
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                optimizer.zero_grad()
                loss = criterion(model(X), y)
                loss.backward()
                optimizer.step()
        self.model_state = copy.deepcopy(model.state_dict())
    
    def send_model_to_helper(self):
        weights = flatten_state_dict(self.model_state.copy())
        self.true_weights = weights
        fed_wt = self.len_dataset/tot_size
        # fed_wt = torch.tensor(fed_wt,dtype=torch.float32)
        weights= fed_wt*weights
        self.split_weights = split_tensor(weights, num_helpers-num_erasures)
        self.split_weights = MDS_CODE @ self.split_weights
        for i in range(num_helpers):
            helpers[i][self.cid] = self.split_weights[i]
            
        for i in range(len(GLOBAL_FAILURE_MATRIX[self.cid])):
            if GLOBAL_FAILURE_MATRIX[self.cid][i]==1:
                helpers[i][self.cid] = None
        

In [ ]:
clients = [Client(cid=i, dataset=client_subsets[i]) for i in range(num_clients)]


In [ ]:
fail_mat = np.array(GLOBAL_FAILURE_MATRIX)
class Master:
    def __init__(self, global_model,device="cpu"):
        set_seed(42)
        self.global_model = global_model
        self.device = device
    
    def aggregate(self):
        pass
    
    def send_model_to_clients(self):
        for  client in clients:
            client.model_state = copy.deepcopy(self.global_model.state_dict())

    def get_model_from_helpers(self):
        final_model = []
        for (failed_key,failed_conns) in GLOBAL_FAILURE_DICT.items():
            true_key = list(range(num_helpers))
            true_key: List
            for val in failed_key:
                true_key.remove(val)
            
            
            helpers_curr = list(map(lambda x: helpers[x],true_key))
            
            fin_li = []
            for h in helpers_curr:
                curr = []
                for conn in failed_conns:
                    curr.append(h[conn])
                fin_li.append(sum(curr))
                currsum = sum(curr)
                payload_dict[f"helper_{h.hid}"].append(payload(currsum))
            fin_li = np.array(fin_li)
            fin = fin_li
            fail_sub = MDS_CODE_NP[np.ix_(true_key,list(range(len(true_key))))]
            fin_mat = np.linalg.inv(fail_sub)@fin
            final_model.append(fin_mat)
        return sum(final_model)
    
    def run_round(self,  epochs=1, batch_size=32, lr=0.01):
        for cli in clients:
            cli.train_local(self.global_model,epochs=epochs,batch_size=batch_size,lr=lr)
            cli.send_model_to_helper()
        final_model = self.get_model_from_helpers()
        fin_state_dict = rebuild_state_dict_from_flat(rebuild_flat_tensor_from_pieces(tuple(map(torch.tensor,final_model))))
        self.global_model.load_state_dict(fin_state_dict)
            
            
    
    def evaluate(self, test_dataset, batch_size=32):
        loader = make_loader(test_dataset, batch_size=batch_size)
        model = self.global_model.to(self.device)
        model.eval()

        total, correct, total_loss = 0, 0, 0
        criterion = nn.CrossEntropyLoss()

        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                outputs = model(X)
                total_loss += criterion(outputs, y).item() * y.size(0)
                correct += (outputs.argmax(1) == y).sum().item()
                total += y.size(0)
        loss = total_loss / total
        accuracy = correct / total
        return loss, accuracy

In [ ]:

fl_acc = []
fl_loss = []

In [ ]:
# global_model = FEMNISTNet()
global_model = LogisticRegression()
master = Master(global_model=global_model)
for round in range(1,NUM_ROUNDS+1):
    print(f"--- Round {round} ---")
    master.run_round(epochs=2, batch_size=32, lr=0.05)
    loss, acc = master.evaluate(test_data)
    master.send_model_to_clients()
    fl_acc.append(acc)
    fl_loss.append(loss)


In [ ]:
print(fl_loss,fl_acc)

In [ ]:
payload_dict

In [ ]:
for key in payload_dict.keys():
    payload_dict[key] = sum(payload_dict[key])

In [ ]:
payload_dict

In [ ]:
for i in range(num_helpers):
    if f'helper_{i}' not in payload_dict.keys():
        payload_dict[i]=0

In [ ]:
csv_file=f'payloads_{num_helpers}_{num_erasures}_{num_clients}_lr.csv'

In [ ]:
import pandas as pd

In [ ]:
import os
if not os.path.isfile(csv_file):
    df = pd.DataFrame(columns=payload_dict.keys())
    df.to_csv(csv_file,index=False)

In [ ]:
entry = pd.DataFrame([dict(payload_dict)])
entry.to_csv(csv_file,mode='a',header=False,index=False)

In [ ]:
print(dict(payload_dict))